In [1]:
# Ensure project root is on sys.path before importing from src
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.applications.resnet50 import preprocess_input
from src.inference import classify_roi
from src.config import get_dataset_root, MODELS_DIR, ROI_DATASET_DIR


In [2]:
# Portable dataset + model paths using src.config
DATASET_ROOT = get_dataset_root()

# Pick the latest .keras model from outputs/models if not explicit
model_candidates = sorted(MODELS_DIR.glob('*.keras'))
if len(model_candidates) == 0:
    raise FileNotFoundError(f'No .keras model files found in: {MODELS_DIR}')
MODEL_PATH = model_candidates[-1]

# ROI dataset (used for ROI-based demos) — falls back to outputs/roi_dataset_v3
ROI_ROOT = ROI_DATASET_DIR

IMG_SIZE = 224
DISPLAY_WIDTH = 900

BEST_THRESHOLD = 0.25

print('DATASET_ROOT =', DATASET_ROOT)
print('MODEL_PATH =', MODEL_PATH)
print('ROI_ROOT =', ROI_ROOT)


DATASET_ROOT = D:\sem-07\EC-7205-computer-vision-image-processing\CA\project\final_dataset
MODEL_PATH = D:\dev\forged-stamp-recognizer\outputs\models\stamp_resnet50_final.keras
ROI_ROOT = D:\dev\forged-stamp-recognizer\outputs\roi_dataset_v3


In [3]:
def show_image(title, image, cmap=None, figsize=(8, 10)):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [4]:
def contour_stamp_score(contour):
    area = cv2.contourArea(contour)

    if area < 80:
        return None

    x, y, w, h = cv2.boundingRect(contour)

    if w == 0 or h == 0:
        return None

    aspect_ratio = w / h
    compactness = min(w, h) / max(w, h)

    perimeter = cv2.arcLength(contour, True)

    if perimeter == 0:
        return None

    circularity = (4 * np.pi * area) / (perimeter**2)

    bbox_area = w * h
    density = area / bbox_area if bbox_area > 0 else 0

    if compactness < 0.45:
        return None

    if density < 0.08:
        return None

    score = 0.45 * compactness + 0.35 * circularity + 0.20 * density

    return {
        "x": int(x),
        "y": int(y),
        "w": int(w),
        "h": int(h),
        "area": float(area),
        "aspect_ratio": float(aspect_ratio),
        "compactness": float(compactness),
        "circularity": float(circularity),
        "density": float(density),
        "score": float(score),
    }

In [5]:
def find_stamp_candidates_by_contours(image_bgr, display_width=900):
    original_h, original_w = image_bgr.shape[:2]

    scale = display_width / original_w
    resized_h = int(original_h * scale)

    image_resized = cv2.resize(image_bgr, (display_width, resized_h))

    image_hsv = cv2.cvtColor(image_resized, cv2.COLOR_BGR2HSV)

    h = image_hsv[:, :, 0]
    s = image_hsv[:, :, 1]
    v = image_hsv[:, :, 2]

    hue_mask = cv2.inRange(h, 90, 170)
    sat_mask = cv2.inRange(s, 25, 255)
    val_mask = cv2.inRange(v, 30, 255)

    mask = cv2.bitwise_and(hue_mask, sat_mask)
    mask = cv2.bitwise_and(mask, val_mask)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

    mask_opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    mask_cleaned = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel_close)

    contours, _ = cv2.findContours(
        mask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for contour in contours:
        result = contour_stamp_score(contour)

        if result is not None:
            candidates.append(result)

    candidates = sorted(candidates, key=lambda item: item["score"], reverse=True)

    return image_resized, mask_cleaned, candidates

In [6]:
def extract_roi_from_candidate_original(
    original_image, resized_image, candidate, pad_factor=1.0
):
    original_h, original_w = original_image.shape[:2]
    resized_h, resized_w = resized_image.shape[:2]

    scale_x = original_w / resized_w
    scale_y = original_h / resized_h

    x, y, w, h = candidate["x"], candidate["y"], candidate["w"], candidate["h"]

    pad_x = int(w * pad_factor)
    pad_y = int(h * pad_factor)

    x1 = max(x - pad_x, 0)
    y1 = max(y - pad_y, 0)
    x2 = min(x + w + pad_x, resized_w)
    y2 = min(y + h + pad_y, resized_h)

    ox1 = int(x1 * scale_x)
    oy1 = int(y1 * scale_y)
    ox2 = int(x2 * scale_x)
    oy2 = int(y2 * scale_y)

    roi_original = original_image[oy1:oy2, ox1:ox2]

    bbox_original = (ox1, oy1, ox2, oy2)

    return roi_original, bbox_original

In [7]:
model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully")
print("Using decision threshold:", BEST_THRESHOLD)

Model loaded successfully
Using decision threshold: 0.25


In [9]:
def preprocess_roi_for_model(roi_bgr):
    roi_rgb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB)

    roi_resized = cv2.resize(
        roi_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA
    )

    roi_array = roi_resized.astype(np.float32)
    roi_array = preprocess_input(roi_array)

    roi_array = np.expand_dims(roi_array, axis=0)

    return roi_array, roi_resized

In [16]:
def run_stamp_inference(image_path):
    start_time = time.time()

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    image_resized, mask_cleaned, candidates = find_stamp_candidates_by_contours(
        image_bgr, display_width=DISPLAY_WIDTH
    )

    if len(candidates) == 0:
        return {"success": False, "reason": "No stamp candidate found"}

    best_candidate = candidates[0]

    roi_bgr, bbox_original = extract_roi_from_candidate_original(
        original_image=image_bgr,
        resized_image=image_resized,
        candidate=best_candidate,
        pad_factor=1.0,
    )

    roi_display_rgb = preprocess_roi_for_model(roi_bgr)[1]

    verdict = classify_roi(model, roi_bgr, decision_threshold=BEST_THRESHOLD)

    forged_probability = float(verdict["raw_prob"])
    predicted_label = verdict["class"].lower()
    confidence = float(verdict["confidence"])

    end_time = time.time()
    inference_time = end_time - start_time

    return {
        "success": True,
        "image_bgr": image_bgr,
        "image_resized": image_resized,
        "mask_cleaned": mask_cleaned,
        "candidates": candidates,
        "best_candidate": best_candidate,
        "roi_bgr": roi_bgr,
        "roi_display_rgb": roi_display_rgb,
        "bbox_original": bbox_original,
        "forged_probability": forged_probability,
        "predicted_label": predicted_label,
        "confidence": confidence,
        "inference_time": inference_time,
    }


In [17]:
sample_path = DATASET_ROOT / "class_1_forged" / "stamp_B" / "Forged_B_300dpi_031.png"

result = run_stamp_inference(sample_path)

print("Success:", result["success"])

if result["success"]:
    print("Prediction:", result["predicted_label"].upper())
    print("Forged probability:", result["forged_probability"])
    print("Confidence:", result["confidence"])
    print("Inference time:", result["inference_time"], "seconds")
else:
    print("Reason:", result["reason"])


ValueError: Input 0 with name 'input_layer_1' of layer 'functional_1' is incompatible with the layer: expected shape=(None, 224, 224, 3), found shape=(1, 878, 877, 3)

In [20]:
if result["success"]:
    image_rgb = cv2.cvtColor(result["image_resized"], cv2.COLOR_BGR2RGB)

    output = result["image_resized"].copy()

    c = result["best_candidate"]

    x, y, w, h = c["x"], c["y"], c["w"], c["h"]

    cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 2)

    output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)

    show_image("Original Document", image_rgb, figsize=(8, 10))

    show_image(
        "HSV + Morphology Mask", result["mask_cleaned"], cmap="gray", figsize=(8, 10)
    )

    show_image("Detected Stamp Candidate", output_rgb, figsize=(8, 10))

    show_image(
        f"Extracted ROI | Prediction: {result['predicted_label'].upper()}",
        result["roi_display_rgb"],
        figsize=(5, 5),
    )

NameError: name 'result' is not defined

In [19]:
if result["success"]:
    print("=" * 50)
    print("STAMP VERIFICATION RESULT")
    print("=" * 50)
    print(f"Input image        : {sample_path.name}")
    print(f"Prediction         : {result['predicted_label'].upper()}")
    print(f"Forged probability : {result['forged_probability']:.4f}")
    print(f"Decision threshold : {BEST_THRESHOLD}")
    print(f"Confidence         : {result['confidence']:.4f}")
    print(f"Inference time     : {result['inference_time']:.4f} seconds")
    print("=" * 50)

NameError: name 'result' is not defined

In [18]:
import pandas as pd


test_images = (
    list((DATASET_ROOT / "class_0_genuine").rglob("*.png"))[:5]
    + list((DATASET_ROOT / "class_1_forged").rglob("*.png"))[:5]
)

batch_results = []

for img_path in test_images:
    result = run_stamp_inference(img_path)

    if result["success"]:
        batch_results.append(
            {
                "image_name": img_path.name,
                "actual_class": (
                    "genuine" if "class_0_genuine" in str(img_path) else "forged"
                ),
                "predicted_class": result["predicted_label"],
                "forged_probability": result["forged_probability"],
                "confidence": result["confidence"],
                "inference_time": result["inference_time"],
            }
        )
    else:
        batch_results.append(
            {
                "image_name": img_path.name,
                "actual_class": (
                    "genuine" if "class_0_genuine" in str(img_path) else "forged"
                ),
                "predicted_class": "failed",
                "forged_probability": None,
                "confidence": None,
                "inference_time": None,
            }
        )

df_batch_results = pd.DataFrame(batch_results)

df_batch_results

ValueError: Input 0 with name 'input_layer_1' of layer 'functional_1' is incompatible with the layer: expected shape=(None, 224, 224, 3), found shape=(1, 663, 678, 3)

In [ ]:
valid_results = df_batch_results[df_batch_results["predicted_class"] != "failed"].copy()

valid_results["correct"] = (
    valid_results["actual_class"] == valid_results["predicted_class"]
)

print("Batch demo accuracy:")
print(valid_results["correct"].mean())

valid_results